# Step 2 — Load SQLCoder 7B + Apply LoRA

Keep Mac plugged in. First run downloads ~14GB.

In [ ]:
import os, torch
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

if torch.backends.mps.is_available():
    DEVICE = 'mps'
elif torch.cuda.is_available():
    DEVICE = 'cuda'
else:
    DEVICE = 'cpu'

MODEL_NAME  = 'defog/sqlcoder-7b-2'
MAX_SEQ_LEN = 512

print(f'Device  : {DEVICE.upper()}')
print(f'PyTorch : {torch.__version__}')

In [ ]:
from transformers import AutoTokenizer

print(f'Loading tokenizer: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
    trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f'Vocab size : {tokenizer.vocab_size:,}')
print(f'Pad token  : {tokenizer.pad_token!r}')
print('✅ Tokenizer ready')

In [ ]:
from transformers import AutoModelForCausalLM

print(f'Loading SQLCoder 7B → {DEVICE.upper()} (float16)')
print('First run downloads ~14GB ...')

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map={'': DEVICE},
    trust_remote_code=True,
)
model.config.use_cache      = False
model.config.pretraining_tp = 1

total = sum(p.numel() for p in model.parameters())
print(f'Parameters : {total/1e9:.2f}B')
print(f'Dtype      : {next(model.parameters()).dtype}')
print(f'Device     : {next(model.parameters()).device}')
print('✅ Base model loaded')

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj'],
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print('✅ LoRA applied — paste output in chat')

In [ ]:
# Quick sanity check — base model generates SQL before fine-tuning
prompt = (
    '### Task\n'
    'Generate a SQL query to answer the following question.\n\n'
    '### Input\n'
    'prompt: How many users are in the users table? '
    'schema: CREATE TABLE users (id INT, name VARCHAR(50));\n\n'
    '### Response\n'
)
inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)
with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
generated = out[0][inputs['input_ids'].shape[1]:]
print('Base model output:')
print(tokenizer.decode(generated, skip_special_tokens=True))
print()
print('✅ Sanity check done — paste full output in chat')